Data
python
train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test  = ["the cat sat on the rug", "the fish sat on the mat"]
Note: lowercase everything, add <s> and </s>, V = all words plus </s> (no <s>), use ln, round to 4 decimals, standard library only.


TASK 1
Tokenization
Write tokenize(sentence) that lowercases, splits on spaces and adds <s> and </s>.
Apply it to every training and test sentence and print the results.
Print the total number of tokens in the training corpus with markers and without markers.
Write build_vocab(sentences) and print V and the sorted vocabulary.


In [1]:

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]

test = ["the cat sat on the rug", "the fish sat on the mat"]


#Task 1.1: Tokenization
def tokenize(sentence):
    words = sentence.lower().split(" ")
    tokens = ["<s>"] + words + ["</s>"]
    return tokens


#Task 1.2: Apply tokenization
train_tokens = [tokenize(sentence) for sentence in train]
test_tokens = [tokenize(sentence) for sentence in test]

print("Training tokens:")
for tokens in train_tokens:
    print(tokens)

print("\nTest tokens:")
for tokens in test_tokens:
    print(tokens)


#Task 1.3: Count tokens
with_markers = sum(len(tokens) for tokens in train_tokens)
without_markers = sum(len(sentence.split(" ")) for sentence in train)

print("\nTokens with markers:", with_markers)
print("Tokens without markers:", without_markers)


#Task 1.4: Build vocabulary
def build_vocab(sentences):
    vocab = set()

    for sentence in sentences:
        words = tokenize(sentence)
        vocab.update(words)

    vocab.discard("<s>")
    return vocab


vocab = build_vocab(train)

print("\nVocabulary size V:", len(vocab))
print("Sorted vocabulary:", sorted(vocab))


Training tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Test tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Tokens with markers: 30
Tokens without markers: 22

Vocabulary size V: 11
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


TASK 2
Generating N-grams
Write get_ngrams(tokens, n) that returns a list of tuples.
Print the unigrams, bigrams and trigrams of the first training sentence.
For every sentence, verify with code that m tokens give m − n + 1 N-grams for n = 1, 2 and 3.
Show what get_ngrams returns when n is larger than the sentence length, and make sure it does not crash.

In [2]:

#Task 2: Generating N-grams

def get_ngrams(tokens, n):
    if n < 1:
        raise ValueError("n must be at least 1")

    ngrams = []

    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i+n]))

    return ngrams


#Print n-grams of the first training sentence
first_sentence = train_tokens[0]

print("Unigrams:", get_ngrams(first_sentence, 1))
print("Bigrams:", get_ngrams(first_sentence, 2))
print("Trigrams:", get_ngrams(first_sentence, 3))


#Verify the n-gram count formula
print("\nChecking n-gram counts:")

for name, sentences in [("Train", train_tokens), ("Test", test_tokens)]:
    for i, tokens in enumerate(sentences, start=1):
        m = len(tokens)

        for n in [1, 2, 3]:
            expected = max(0, m - n + 1)
            actual = len(get_ngrams(tokens, n))

            print(name, i, "n =", n,
                  "expected =", expected,
                  "actual =", actual,
                  "PASS" if expected == actual else "FAIL")


#Test n larger than sentence length
print("\nWhen n is too large:")
print(get_ngrams(["the", "cat", "sat"], 5))


Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

Checking n-gram counts:
Train 1 n = 1 expected = 8 actual = 8 PASS
Train 1 n = 2 expected = 7 actual = 7 PASS
Train 1 n = 3 expected = 6 actual = 6 PASS
Train 2 n = 1 expected = 7 actual = 7 PASS
Train 2 n = 2 expected = 6 actual = 6 PASS
Train 2 n = 3 expected = 5 actual = 5 PASS
Train 3 n = 1 expected = 8 actual = 8 PASS
Train 3 n = 2 expected = 7 actual = 7 PASS
Train 3 n = 3 expected = 6 actual = 6 PASS
Train 4 n = 1 expected = 7 actual = 7 PASS
Train 4 n = 2 expected = 6 actual = 6 PASS
Train 4 n = 3 expected = 5 actual = 5 PASS
Test 1 n = 1 expected = 8 actual = 8 PASS
Test 1 n = 2 expected = 7 actual = 7 PASS
Test 1 n = 3 exp

TASK 3
Counting N-grams
Use collections.Counter to build unigram, bigram and trigram counts for the training corpus.
Print each count table sorted from highest to lowest count.
Print the 5 most common bigrams and the number of distinct unigrams, bigrams and trigrams.
Write count_of(ngram) that returns 0 for an unseen N-gram instead of raising an error. Test it on one seen and one unseen bigram.


In [3]:

from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]

#Convert each sentence to lowercase and add sentence markers
def tokenize(sentence):
    words = sentence.lower().split(" ")
    tokens = ["<s>"] + words + ["</s>"]
    return tokens

#Tokenize the training corpus
train_tokens = [tokenize(sentence) for sentence in train]

#Generate n-grams
def get_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i+n]))
    return ngrams

#Count unigrams, bigrams and trigrams
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))

#Print count tables from highest to lowest count
print("Unigram Counts:")
for ngram, count in unigram_counts.most_common():
    print(ngram, ":", count)

print("\nBigram Counts:")
for ngram, count in bigram_counts.most_common():
    print(ngram, ":", count)

print("\nTrigram Counts:")
for ngram, count in trigram_counts.most_common():
    print(ngram, ":", count)

#Print the 5 most common bigrams
print("\nTop 5 Most Common Bigrams:")
for ngram, count in bigram_counts.most_common(5):
    print(ngram, ":", count)

#Print the number of distinct n-grams
print("\nDistinct Unigrams:", len(unigram_counts))
print("Distinct Bigrams:", len(bigram_counts))
print("Distinct Trigrams:", len(trigram_counts))

#Return 0 if an n-gram has not been seen
def count_of(ngram):
    return unigram_counts.get(ngram, 0) if len(ngram) == 1 else (
        bigram_counts.get(ngram, 0) if len(ngram) == 2 else
        trigram_counts.get(ngram, 0) if len(ngram) == 3 else 0
    )

#Test one seen and one unseen bigram
print("\nSeen bigram:", count_of(("the", "cat")))
print("Unseen bigram:", count_of(("cat", "the")))


Unigram Counts:
('the',) : 8
('<s>',) : 4
('</s>',) : 4
('cat',) : 2
('sat',) : 2
('on',) : 2
('ate',) : 2
('dog',) : 2
('mat',) : 1
('fish',) : 1
('rug',) : 1
('bone',) : 1

Bigram Counts:
('<s>', 'the') : 4
('the', 'cat') : 2
('sat', 'on') : 2
('on', 'the') : 2
('ate', 'the') : 2
('the', 'dog') : 2
('cat', 'sat') : 1
('the', 'mat') : 1
('mat', '</s>') : 1
('cat', 'ate') : 1
('the', 'fish') : 1
('fish', '</s>') : 1
('dog', 'sat') : 1
('the', 'rug') : 1
('rug', '</s>') : 1
('dog', 'ate') : 1
('the', 'bone') : 1
('bone', '</s>') : 1

Trigram Counts:
('<s>', 'the', 'cat') : 2
('sat', 'on', 'the') : 2
('<s>', 'the', 'dog') : 2
('the', 'cat', 'sat') : 1
('cat', 'sat', 'on') : 1
('on', 'the', 'mat') : 1
('the', 'mat', '</s>') : 1
('the', 'cat', 'ate') : 1
('cat', 'ate', 'the') : 1
('ate', 'the', 'fish') : 1
('the', 'fish', '</s>') : 1
('the', 'dog', 'sat') : 1
('dog', 'sat', 'on') : 1
('on', 'the', 'rug') : 1
('the', 'rug', '</s>') : 1
('the', 'dog', 'ate') : 1
('dog', 'ate', 'the') : 1
('a

`TASK 4
Unigram model
Write unigram_prob(w) using MLE, P(w) = C(w) / N.
Print all unigram probabilities as a table and assert that they sum to 1.
Write unigram_sentence_prob(sentence) and print the result for both test sentences.
Use itertools.permutations on the words of the first test sentence and show that different word orders can give the same unigram probability.`


In [4]:

from collections import Counter
from itertools import permutations

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]

test = ["the cat sat on the rug", "the fish sat on the mat"]

#Step 1: Tokenize sentences
def tokenize(sentence):
    words = sentence.lower().split()
    return ["<s>"] + words + ["</s>"]

train_tokens = [tokenize(sentence) for sentence in train]
test_tokens = [tokenize(sentence) for sentence in test]

#Step 2: Count all unigrams in the training corpus
unigram_counts = Counter()

for tokens in train_tokens:
    for word in tokens:
        unigram_counts[word] += 1

#Step 3: Calculate total number of unigram tokens
N = sum(unigram_counts.values())

#Step 4: Calculate unigram probability using MLE
def unigram_prob(w):
    return unigram_counts.get(w, 0) / N

#Step 5: Print all unigram probabilities
print("Unigram Probability Table")
print("-------------------------")
print("Word\t\tCount\tProbability")

for word, count in sorted(unigram_counts.items()):
    print(f"{word:<10}\t{count}\t{unigram_prob(word):.4f}")

#Check that all unigram probabilities sum to 1
total_probability = sum(unigram_prob(word) for word in unigram_counts)
print("\nSum of probabilities:", round(total_probability, 10))

assert abs(total_probability - 1.0) < 1e-9
print("Assertion passed: probabilities sum to 1.")

#Step 6: Calculate the probability of a complete sentence
def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = 1.0

    for word in tokens:
        probability *= unigram_prob(word)

    return probability

#Print probability of both test sentences
print("\nTest Sentence Probabilities")

for sentence in test:
    print(sentence, "=", f"{unigram_sentence_prob(sentence):.10g}")

#Step 7: Permute the words of the first test sentence
words = test[0].lower().split()

print("\nPermutation Test")

for word_order in list(permutations(words))[:5]:
    sentence = " ".join(word_order)
    print(sentence, "=", f"{unigram_sentence_prob(sentence):.10g}")


Unigram Probability Table
-------------------------
Word		Count	Probability
</s>      	4	0.1333
<s>       	4	0.1333
ate       	2	0.0667
bone      	1	0.0333
cat       	2	0.0667
dog       	2	0.0667
fish      	1	0.0333
mat       	1	0.0333
on        	2	0.0667
rug       	1	0.0333
sat       	2	0.0667
the       	8	0.2667

Sum of probabilities: 1.0
Assertion passed: probabilities sum to 1.

Test Sentence Probabilities
the cat sat on the rug = 1.248590154e-08
the fish sat on the mat = 6.24295077e-09

Permutation Test
the cat sat on the rug = 1.248590154e-08
the cat sat on rug the = 1.248590154e-08
the cat sat the on rug = 1.248590154e-08
the cat sat the rug on = 1.248590154e-08
the cat sat rug on the = 1.248590154e-08


TASK 5
Bigram model (MLE)
Write bigram_prob(prev, w) using P(w | v) = C(v, w) / C(v).
Write next_word_distribution(prev) that returns a dictionary of next words and probabilities. Print it for “the” and assert that it sums to 1.
Write bigram_sentence_prob(sentence) using the chain rule and print the result for both test sentences.
Write predict_next(prev) that returns the most probable next word (break ties alphabetically) and test it on at least five different words.


In [5]:

from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]

test = ["the cat sat on the rug", "the fish sat on the mat"]

#Step 1: Tokenize sentences
def tokenize(sentence):
    words = sentence.lower().split()
    return ["<s>"] + words + ["</s>"]

train_tokens = [tokenize(sentence) for sentence in train]
test_tokens = [tokenize(sentence) for sentence in test]

#Step 2: Count bigrams and their previous-word contexts
bigram_counts = Counter()
context_counts = Counter()

for tokens in train_tokens:
    for i in range(len(tokens) - 1):
        prev = tokens[i]
        word = tokens[i + 1]

        bigram_counts[(prev, word)] += 1
        context_counts[prev] += 1

#Step 3: Calculate bigram probability using MLE
def bigram_prob(prev, w):
    if context_counts[prev] == 0:
        return 0.0

    return bigram_counts[(prev, w)] / context_counts[prev]

#Step 4: Get probabilities of all next words after prev
def next_word_distribution(prev):
    distribution = {}

    for (previous, word), count in bigram_counts.items():
        if previous == prev:
            distribution[word] = count / context_counts[prev]

    return distribution

#Print the next-word distribution for "the"
distribution = next_word_distribution("the")

print("Next-word distribution for 'the':")
for word, probability in sorted(distribution.items()):
    print(f"{word}: {probability:.4f}")

#Check that the probabilities sum to 1
assert abs(sum(distribution.values()) - 1.0) < 1e-9
print("Assertion passed: probabilities sum to 1.")

#Step 5: Calculate sentence probability using the chain rule
def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    probability = 1.0

    for i in range(len(tokens) - 1):
        probability *= bigram_prob(tokens[i], tokens[i + 1])

    return probability

#Print probabilities for both test sentences
print("\nTest sentence probabilities:")
for sentence in test:
    print(sentence, "=", f"{bigram_sentence_prob(sentence):.10g}")

#Step 6: Predict the most probable next word
#Break ties alphabetically
def predict_next(prev):
    distribution = next_word_distribution(prev)

    if not distribution:
        return None

    return sorted(distribution, key=lambda word: (-distribution[word], word))[0]

#Test on five different previous words
print("\nNext-word predictions:")
for prev in ["the", "cat", "dog", "sat", "on"]:
    print(prev, "->", predict_next(prev))


Next-word distribution for 'the':
bone: 0.1250
cat: 0.2500
dog: 0.2500
fish: 0.1250
mat: 0.1250
rug: 0.1250
Assertion passed: probabilities sum to 1.

Test sentence probabilities:
the cat sat on the rug = 0.015625
the fish sat on the mat = 0

Next-word predictions:
the -> cat
cat -> ate
dog -> ate
sat -> on
on -> the


TASK 6
Trigram model
Write trigram_prob(w1, w2, w3) using P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2).
Write trigram_sentence_prob(sentence). Use the bigram model for the first word given <s> and the trigram model for all other words. Print the result for both test sentences.
Print the number of distinct unigrams, bigrams and trigrams in a small table.
For each test sentence, list its unseen bigrams and its unseen trigrams, then print which model has more unseen N-grams.


In [ ]:

from collections import Counter

train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]

test = ["the cat sat on the rug", "the fish sat on the mat"]

#Step 1: Tokenize sentences
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

train_tokens = [tokenize(s) for s in train]

#Step 2: Count unigrams, bigrams and trigrams
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update((tokens[i],) for i in range(len(tokens)))

    bigram_counts.update(
        tuple(tokens[i:i+2]) for i in range(len(tokens)-1)
    )

    trigram_counts.update(
        tuple(tokens[i:i+3]) for i in range(len(tokens)-2)
    )

#Step 3: Bigram probability, needed for the first word
def bigram_prob(prev, word):
    denominator = sum(
        count for (v, w), count in bigram_counts.items()
        if v == prev
    )

    if denominator == 0:
        return 0.0

    return bigram_counts[(prev, word)] / denominator

#Step 4: Trigram probability using MLE
def trigram_prob(w1, w2, w3):
    denominator = bigram_counts[(w1, w2)]

    if denominator == 0:
        return 0.0

    return trigram_counts[(w1, w2, w3)] / denominator

#Step 5: Calculate sentence probability
#Use bigram for the first word and trigrams for all remaining tokens
def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    probability = bigram_prob(tokens[0], tokens[1])

    for i in range(2, len(tokens)):
        probability *= trigram_prob(
            tokens[i-2], tokens[i-1], tokens[i]
        )

    return probability

print("Trigram Sentence Probabilities:")
for sentence in test:
    print(sentence, "=", f"{trigram_sentence_prob(sentence):.10g}")

#Step 6: Print distinct n-gram counts
print("\nDistinct N-gram Counts")
print("----------------------")
print(f"{'N-gram':<12}{'Distinct count'}")
print(f"{'Unigrams':<12}{len(unigram_counts)}")
print(f"{'Bigrams':<12}{len(bigram_counts)}")
print(f"{'Trigrams':<12}{len(trigram_counts)}")

#Step 7: Find unseen bigrams and trigrams in each test sentence
for sentence in test:
    tokens = tokenize(sentence)

    bigrams = [
        tuple(tokens[i:i+2])
        for i in range(len(tokens)-1)
    ]

    trigrams = [
        tuple(tokens[i:i+3])
        for i in range(len(tokens)-2)
    ]

    unseen_bigrams = [
        bg for bg in bigrams if bigram_counts[bg] == 0
    ]

    unseen_trigrams = [
        tg for tg in trigrams if trigram_counts[tg] == 0
    ]

    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bigrams)
    print("Unseen trigrams:", unseen_trigrams)

    if len(unseen_bigrams) > len(unseen_trigrams):
        print("More unseen N-grams: Bigram model")
    elif len(unseen_trigrams) > len(unseen_bigrams):
        print("More unseen N-grams: Trigram model")
    else:
        print("More unseen N-grams: Tie")


TASK 7
Zero-probability problem
Write unseen_bigrams(sentence) that returns every bigram of the sentence that never occurs in training.
Run it on both test sentences and print the bigram MLE probability of each. Confirm that one of them is 0.
Count how many of the possible word pairs (words only, markers excluded) never occur in training, and print this as a percentage.
Write safe_log(p) that returns negative infinity for p = 0 instead of raising a math error, and use it to compute the log probability of the second test sentence.


TASK 8
Smoothing
Write bigram_prob_addone(prev, w) using (C(v, w) + 1) / (C(v) + V).
Print P(sat | fish) and P(cat | the) under both MLE and add-one smoothing, side by side.
Loop over every possible previous word and assert that its smoothed next-word probabilities over all V words sum to 1.
Write bigram_prob_addk(prev, w, k) and print P(sat | fish) for k = 1, 0.5, 0.1 and 0.01 in a table.


TASK 9
Perplexity and model comparison
Write perplexity(sentences, prob_fn) that returns infinity when any probability is 0.
Compute the perplexity of the two test sentences together for the bigram MLE model, the bigram add-one model and the unigram add-one model, where P(w) = (C(w) + 1) / (N + V). Print one results table.
Compute the perplexity of the add-k bigram model for k = 1, 0.5, 0.1 and 0.01. Print the table and the best k.
Train only on the first three training sentences (build V from all four) and pick the best k using the fourth sentence as validation data. Report the test perplexity with that k, and compare it with the best k found directly on the test set.
